In [11]:
import pandas as pd
import altair as alt
from google.colab import files

uploaded = files.upload()
nombre = list(uploaded.keys())[0]
df = pd.read_csv(nombre)
df.head()

Saving base_datos_oficial_ordenada.csv to base_datos_oficial_ordenada (1).csv


,nombres_nominados_nominadas,genero,lugar_de_nacimiento,nacionalidad_1,nacionalidad_2,raza_o_etnia,su_debut_fue_estadounidense,titulo_original_pelicula_debut,categoria_nominacion_oscar,fue_ganador_o_ganadora,ano_ceremonia,titulo_original_pelicula_nominada,ano_pelicula_nominada,idioma_original,generos,companias_de_produccion,paises_de_produccion,idiomas_hablados
0,Richard Barthelmess,Masculino,"Nueva York, Estados Unidos",Estadounidense,NaN,Blanca,Sí,War Brides,Mejor actor,No,1929,The Noose,1927,Inglés,"Drama, Crimen",First National Pictures,Estados Unidos,Inglés
1,Richard Barthelmess,Masculino,"Nueva York, Estados Unidos",Estadounidense,NaN,Blanca,No,War Brides,Mejor actor,No,1929,The Patent Leather Kid,1927,Inglés,"Guerra, Drama, Romance",First National Pictures,Estados Unidos,Inglés
2,Emil Jannings,Masculino,"Rorschach, Suiza",Suiza,Alemana,Blanca,No,Arme Eva,Mejor actor,Sí,1929,The Last Command,1927,Inglés,"Drama, Bélica",Paramount Pictures,Estados Unidos,Inglés
3,Emil Jannings,Masculino,"Rorschach, Suiza",Suiza,Alemana,Blanca,No,Arme Eva,Mejor actor,No,1929,The Way of All Flesh,1927,Inglés,Drama,Paramount,Estados Unidos,Inglés
4,Louise Dresser,Femenino,"Evansville, Estados Unidos",Estadounidense,NaN,Blanca,Sí,The Glory of Clementina,Mejor actriz,No,1929,A Ship Comes In,1927,Inglés,Drama,"DeMille Pictures Corporation, Pathé Exchange",Estados Unidos,Inglés


In [13]:
#Una fila por película
peliculas = (
    df.drop_duplicates(subset=["titulo_original_pelicula_nominada", "ano_pelicula_nominada"])
      [["titulo_original_pelicula_nominada", "ano_pelicula_nominada",
        "generos", "paises_de_produccion"]]
      .copy()
)

# Origen: estadounidense vs extranjera, todo lo que no sea Estadounidense es considerada extranjera
peliculas["origen"] = peliculas["paises_de_produccion"].fillna("").apply(
    lambda x: "Estadounidense" if "Estados Unidos" in x else "Extranjera"
)

# Década
peliculas["decada"] = (peliculas["ano_pelicula_nominada"] // 10 * 10).astype(int).astype(str) + "s"

# Separar géneros, para que sea uno por fila y limpiar
gen = peliculas.assign(genero=peliculas["generos"].fillna("").str.split(","))
gen = gen.explode("genero")
gen["genero"] = gen["genero"].str.strip().replace({"Bélica": "Guerra", "": None})
gen = gen.dropna(subset=["genero"])

# Conteo y porcentaje de películas de cada década que tienen ese género
total_decada = (peliculas.groupby(["origen", "decada"]).size()
                .rename("total_peliculas").reset_index())

conteo = (gen.groupby(["origen", "decada", "genero"]).size()
          .rename("n").reset_index()
          .merge(total_decada, on=["origen", "decada"]))
conteo["porcentaje"] = conteo["n"] / conteo["total_peliculas"] * 100

print(peliculas["origen"].value_counts())

origen
Estadounidense    746
Extranjera        124
Name: count, dtype: int64


In [14]:
def grafico_tendencia(origen, top_n=6, color_scheme="category10"):
    datos = conteo[conteo["origen"] == origen]

    # Top N géneros del grupo
    top = (datos.groupby("genero")["n"].sum()
           .nlargest(top_n).index.tolist())

    # Rellenar con 0 las décadas sin ese género
    decadas = sorted(datos["decada"].unique())
    grilla = pd.MultiIndex.from_product([decadas, top], names=["decada", "genero"]).to_frame(index=False)
    d = (grilla.merge(datos[["decada", "genero", "n", "porcentaje", "total_peliculas"]],
                      on=["decada", "genero"], how="left")
              .fillna({"n": 0, "porcentaje": 0}))
    d["total_peliculas"] = d["decada"].map(
        datos.drop_duplicates("decada").set_index("decada")["total_peliculas"])

    seleccion = alt.selection_point(fields=["genero"], bind="legend")

    return (
        alt.Chart(d)
        .mark_line(point=True, strokeWidth=2.5)
        .encode(
            x=alt.X("decada:O", title="Década de la película"),
            y=alt.Y("porcentaje:Q", title="% de películas con ese género"),
            color=alt.Color("genero:N", title="Género",
                            scale=alt.Scale(scheme=color_scheme)),
            opacity=alt.condition(seleccion, alt.value(1), alt.value(0.15)),
            tooltip=[alt.Tooltip("decada:O", title="Década"),
                     alt.Tooltip("genero:N", title="Género"),
                     alt.Tooltip("n:Q", title="N° películas"),
                     alt.Tooltip("total_peliculas:Q", title="Total películas década"),
                     alt.Tooltip("porcentaje:Q", title="%", format=".1f")],
        )
        .add_params(seleccion)
        .properties(width=650, height=350,
                    title=f"Géneros predominantes en películas {origen.lower()}s nominadas")
    )

g_usa = grafico_tendencia("Estadounidense")
g_ext = grafico_tendencia("Extranjera")

g_usa

alt.Chart(...)

In [20]:
g_usa.save("nombre.html")

In [15]:
g_ext   #Visualización tendencias películas extranjeras

alt.Chart(...)

In [16]:
(g_usa & g_ext).resolve_scale(color="independent")   #Ambas visualizaciones en una imagen

alt.VConcatChart(...)

In [17]:
!pip install -q vl-convert-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 41.8 MB/s eta 0:00:00


In [21]:
from google.colab import files

def guardar_png(grafico, nombre, escala=3):
    grafico.save(f"{nombre}.png", scale_factor=escala)
    files.download(f"{nombre}.png")

guardar_png(g_usa, "generos_estadounidenses")
guardar_png(g_ext, "generos_extranjeras")

guardar_png((g_usa & g_ext).resolve_scale(color="independent"), "generos_comparacion")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>